In [3]:
import pandas as pd


In [4]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Cargar los datasets

df_delitos = pd.read_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/interim/delitos_propiedad_departamento_anual.csv")
df_geo = pd.read_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/raw/prov_departamentos_nombres.csv")

print("df_delitos:", df_delitos.shape)
print("df_geo:", df_geo.shape)

print("\nColumnas df_delitos:")
print(df_delitos.columns.tolist())

print("\nColumnas df_geo:")
print(df_geo.columns.tolist())

print(df_delitos.head())
df_geo.head()

df_delitos: (13342, 7)
df_geo: (527, 5)

Columnas df_delitos:
['provincia_id', 'provincia_nombre', 'departamento_id', 'departamento_nombre', 'anio', 'delitos_propiedad_hechos', 'delitos_propiedad_victimas']

Columnas df_geo:
['redcode', 'nprov_0', 'ndpto_1', 'redcoden_2', 'Unnamed: 4']
   provincia_id                 provincia_nombre  departamento_id  \
0             2  Ciudad Autónoma de Buenos Aires             2001   
1             2  Ciudad Autónoma de Buenos Aires             2001   
2             2  Ciudad Autónoma de Buenos Aires             2001   
3             2  Ciudad Autónoma de Buenos Aires             2001   
4             2  Ciudad Autónoma de Buenos Aires             2001   

  departamento_nombre  anio  delitos_propiedad_hechos  \
0            Comuna 1  2000                         0   
1            Comuna 1  2001                         0   
2            Comuna 1  2002                         0   
3            Comuna 1  2003                         0   
4            

,redcode,nprov_0,ndpto_1,redcoden_2,Unnamed: 4
0,2007,Caba,Comuna 1,2007,NaN
1,2014,Caba,Comuna 2,2014,NaN
2,2021,Caba,Comuna 3,2021,NaN
3,2028,Caba,Comuna 4,2028,NaN
4,2035,Caba,Comuna 5,2035,NaN


In [6]:
# Seleccionar solo las columnas necesarias y renombrarlas para facilitar el merge
df_geo = df_geo[["redcode", "nprov_0", "ndpto_1"]]

df_geo = df_geo.rename(columns={
    "redcode": "departamento_id_geo",
    "nprov_0": "provincia_geo",
    "ndpto_1": "departamento_geo"
})

df_geo.head()

,departamento_id_geo,provincia_geo,departamento_geo
0,2007,Caba,Comuna 1
1,2014,Caba,Comuna 2
2,2021,Caba,Comuna 3
3,2028,Caba,Comuna 4
4,2035,Caba,Comuna 5


In [7]:
# Normalizar los nombres de provincias y departamentos para facilitar el merge
df_delitos["provincia_nombre"] = df_delitos["provincia_nombre"].str.lower().str.strip()
df_delitos["departamento_nombre"] = df_delitos["departamento_nombre"].str.lower().str.strip()

df_geo["provincia_geo"] = df_geo["provincia_geo"].str.lower().str.strip()
df_geo["departamento_geo"] = df_geo["departamento_geo"].str.lower().str.strip()

In [8]:
# Realizar el merge entre ambos datasets
df_check = df_delitos.merge(
    df_geo,
    left_on=["provincia_nombre", "departamento_nombre"],
    right_on=["provincia_geo", "departamento_geo"],
    how="left"
)

In [9]:
print (df_check.shape)
df_check.head()

(13342, 10)


,provincia_id,provincia_nombre,departamento_id,departamento_nombre,anio,delitos_propiedad_hechos,delitos_propiedad_victimas,departamento_id_geo,provincia_geo,departamento_geo
0,2,ciudad autónoma de buenos aires,2001,comuna 1,2000,0,0.0,NaN,NaN,NaN
1,2,ciudad autónoma de buenos aires,2001,comuna 1,2001,0,0.0,NaN,NaN,NaN
2,2,ciudad autónoma de buenos aires,2001,comuna 1,2002,0,0.0,NaN,NaN,NaN
3,2,ciudad autónoma de buenos aires,2001,comuna 1,2003,0,0.0,NaN,NaN,NaN
4,2,ciudad autónoma de buenos aires,2001,comuna 1,2004,0,0.0,NaN,NaN,NaN


In [10]:
# Identificar filas sin match
import unicodedata

def normalizar_texto(s):
    if pd.isna(s):
        return s
    s = str(s).lower().strip()
    s = ''.join(
        c for c in unicodedata.normalize('NFKD', s)
        if not unicodedata.combining(c)
    )
    s = ' '.join(s.split())
    return s

# normalizar columnas
df_delitos["provincia_key"] = df_delitos["provincia_nombre"].apply(normalizar_texto)
df_delitos["departamento_key"] = df_delitos["departamento_nombre"].apply(normalizar_texto)

df_geo["provincia_key"] = df_geo["provincia_geo"].apply(normalizar_texto)
df_geo["departamento_key"] = df_geo["departamento_geo"].apply(normalizar_texto)

# armonizar provincias conocidas
df_delitos["provincia_key"] = df_delitos["provincia_key"].replace({
    "ciudad autonoma de buenos aires": "caba",
    "tierra del fuego, antartida e islas del atlantico sur": "tierra del fuego"
})

df_geo["provincia_key"] = df_geo["provincia_key"].replace({
    "ciudad autonoma de buenos aires": "caba",
    "tierra del fuego, antartida e islas del atlantico sur": "tierra del fuego"
})

In [11]:
# Rehacer el merge con las claves normalizadas
df_check = df_delitos.merge(
    df_geo,
    on=["provincia_key", "departamento_key"],
    how="left",
    suffixes=("", "_geo")
)

In [12]:
# Verificar filas sin match
df_check[df_check["departamento_id_geo"].isna()][
    ["provincia_nombre", "departamento_nombre"]
].drop_duplicates()

,provincia_nombre,departamento_nombre
375,ciudad autónoma de buenos aires,departamento sin determinar
852,buenos aires,cañuelas
1027,buenos aires,coronel de marina l. rosales
1302,buenos aires,esteban echeverría
1902,buenos aires,ituzaingó
...,...,...
12442,santiago del estero,juan f. ibarra
12792,santiago del estero,departamento sin determinar
12817,tucumán,burruyacú
13242,tucumán,departamento sin determinar


In [13]:
# Analizar filas sin match
faltantes = df_check[df_check["departamento_id_geo"].isna()][
    ["provincia_nombre", "departamento_nombre", "provincia_key", "departamento_key"]
].drop_duplicates()

sin_determinar = faltantes[
    faltantes["departamento_key"] == "departamento sin determinar"
]

no_macheados_reales = faltantes[
    faltantes["departamento_key"] != "departamento sin determinar"
]

print("Sin determinar:", sin_determinar.shape)
print("No macheados reales:", no_macheados_reales.shape)

no_macheados_reales.sort_values(["provincia_nombre", "departamento_nombre"])

Sin determinar: (24, 4)
No macheados reales: (41, 4)


,provincia_nombre,departamento_nombre,provincia_key,departamento_key
852,buenos aires,cañuelas,buenos aires,canuelas
1027,buenos aires,coronel de marina l. rosales,buenos aires,coronel de marina l. rosales
1302,buenos aires,esteban echeverría,buenos aires,esteban echeverria
1902,buenos aires,ituzaingó,buenos aires,ituzaingo
1927,buenos aires,josé c. paz,buenos aires,jose c. paz
2192,buenos aires,lobería,buenos aires,loberia
2267,buenos aires,luján,buenos aires,lujan
2642,buenos aires,olavarría,buenos aires,olavarria
3890,catamarca,belén,catamarca,belen
3915,catamarca,capayán,catamarca,capayan


In [14]:
# Eliminar filas sin determinar
df_delitos = df_delitos[
    df_delitos["departamento_key"] != "departamento sin determinar"
].copy()

df_delitos.shape

(12742, 9)

In [15]:
# Rehacer el merge con las claves normalizadas
df_check = df_delitos.merge(
    df_geo,
    on=["provincia_key", "departamento_key"],
    how="left"
)

In [16]:
# Verificar filas sin match
faltantes_finales = df_check[df_check["departamento_id_geo"].isna()][
    ["provincia_nombre", "departamento_nombre"]
].drop_duplicates()

print("Faltantes finales:", faltantes_finales.shape)
faltantes_finales.sort_values(["provincia_nombre", "departamento_nombre"])

Faltantes finales: (41, 2)


,provincia_nombre,departamento_nombre
827,buenos aires,cañuelas
1002,buenos aires,coronel de marina l. rosales
1277,buenos aires,esteban echeverría
1877,buenos aires,ituzaingó
1902,buenos aires,josé c. paz
2167,buenos aires,lobería
2242,buenos aires,luján
2617,buenos aires,olavarría
3840,catamarca,belén
3865,catamarca,capayán


In [17]:
# Importamos la función que permite buscar strings parecidos
from difflib import get_close_matches

# Lista vacía donde vamos a guardar todas las sugerencias
sugerencias = []

# Recorremos TODOS los faltantes (no solo los primeros 10)
for _, row in faltantes.iterrows():

    # Tomamos la provincia normalizada del dataset de delitos
    prov = row["provincia_key"]

    # Tomamos el nombre del departamento normalizado del dataset de delitos
    dpto = row["departamento_key"]

    # Filtramos df_geo para quedarnos SOLO con esa provincia
    # y extraemos la lista de nombres de departamentos posibles
    candidatos = df_geo[
        df_geo["provincia_key"] == prov
    ]["departamento_key"].dropna().unique().tolist()

    # Buscamos coincidencias aproximadas entre el nombre de delitos y los de df_geo
    # n=3 → máximo 3 sugerencias
    # cutoff=0.6 → nivel mínimo de similitud (0 a 1)
    matches = get_close_matches(dpto, candidatos, n=3, cutoff=0.8)

    # Guardamos la información en un diccionario
    sugerencias.append({
        "provincia_nombre": row["provincia_nombre"],
        "departamento_nombre": row["departamento_nombre"],
        "departamento_key": dpto,
        "sugerencia_1": matches[0] if len(matches) > 0 else None,
        "sugerencia_2": matches[1] if len(matches) > 1 else None,
        "sugerencia_3": matches[2] if len(matches) > 2 else None,
    })

# Convertimos la lista de diccionarios en un DataFrame
df_sugerencias = pd.DataFrame(sugerencias)

# Ordenamos para que sea más fácil de leer
df_sugerencias = df_sugerencias.sort_values(
    ["provincia_nombre", "departamento_nombre"]
)

# Mostramos el resultado
df_sugerencias.head(25)

,provincia_nombre,departamento_nombre,departamento_key,sugerencia_1,sugerencia_2,sugerencia_3
1,buenos aires,cañuelas,canuelas,ca uelas,None,None
2,buenos aires,coronel de marina l. rosales,coronel de marina l. rosales,coronel de marina leonardo rosales,None,None
9,buenos aires,departamento sin determinar,departamento sin determinar,None,None,None
3,buenos aires,esteban echeverría,esteban echeverria,esteban echeverrka,None,None
4,buenos aires,ituzaingó,ituzaingo,ituzaingˇ,None,None
5,buenos aires,josé c. paz,jose c. paz,jost c. paz,None,None
6,buenos aires,lobería,loberia,loberka,None,None
7,buenos aires,luján,lujan,lujsn,None,None
8,buenos aires,olavarría,olavarria,olavarrka,None,None
10,catamarca,belén,belen,belun,None,None


In [18]:
# Construir diccionario automático usando la mejor sugerencia encontrada
correcciones_auto = {
    row["departamento_key"]: row["sugerencia_1"]
    for _, row in df_sugerencias.iterrows()
    if pd.notna(row["sugerencia_1"])
}

In [19]:
# Aplicar correcciones automáticas
df_delitos["departamento_key"] = df_delitos["departamento_key"].replace(correcciones_auto)

In [20]:
# Rehacer el merge con las claves normalizadas
df_check = df_delitos.merge(
    df_geo,
    on=["provincia_key", "departamento_key"],
    how="left"
)



In [21]:
# Verificar filas sin match
faltantes_finales = df_check[df_check["departamento_id_geo"].isna()][
    ["provincia_nombre", "departamento_nombre"]
].drop_duplicates()

print("Faltantes finales:", faltantes_finales.shape)
faltantes_finales.sort_values(["provincia_nombre", "departamento_nombre"])

Faltantes finales: (7, 2)


,provincia_nombre,departamento_nombre
7464,la pampa,centro (santa rosa)
7486,la pampa,norte (general pico)
7530,la pampa,oeste (25 de mayo)
7508,la pampa,sur (general acha)
7792,la rioja,general angel v. peñaloza
7892,la rioja,general ocampo
10892,san luis,la capital


In [22]:
# Analizar caso específico de La Pampa
df_geo[df_geo["provincia_key"] == "la pampa"][
    ["provincia_geo", "departamento_geo", "departamento_key"]
].sort_values("departamento_geo")

,provincia_geo,departamento_geo,departamento_key
299,la pampa,atreucó,atreuco
300,la pampa,caleu caleu,caleu caleu
301,la pampa,capital,capital
302,la pampa,catriló,catrilo
305,la pampa,chalileo,chalileo
306,la pampa,chapaleufú,chapaleufu
307,la pampa,chical co,chical co
303,la pampa,conhelo,conhelo
304,la pampa,curacó,curaco
308,la pampa,guatraché,guatrache


In [23]:
#  Analizar departamentos de La Pampa en el dataset de delitos
df_delitos[
    df_delitos["provincia_nombre"] == "la pampa"
]["departamento_nombre"].drop_duplicates().sort_values()

,departamento_nombre
7802,atreucó
7805,caleu caleu
7808,capital
7811,catriló
7714,centro (santa rosa)
7820,chalileo
7823,chapaleufú
7826,chicalcó
7814,conhelo
7817,curacó


# cerrando dataset base para tasas

In [24]:
# Conclusión: no hay forma de hacer un match correcto para estos casos, por lo que se van a excluir de la base de tasas (pero se mantienen en la base de delitos para futuros análisis)
df_tasas_base = df_check.copy()

df_tasas_base = df_tasas_base[
    df_tasas_base["departamento_key"] != "departamento sin determinar"
].copy()

casos_excluir_tasas = [
    ("la pampa", "centro (santa rosa)"),
    ("la pampa", "norte (general pico)"),
    ("la pampa", "oeste (25 de mayo)"),
    ("la pampa", "sur (general acha)"),
    ("san luis", "la capital"),
]

mask_lp = df_tasas_base.apply(
    lambda row: (row["provincia_key"], row["departamento_key"]) in casos_excluir_tasas,
    axis=1
)

df_tasas_base = df_tasas_base[~mask_lp].copy()

df_tasas_base = df_tasas_base[
    df_tasas_base["departamento_id_geo"].notna()
].copy()

print("Base para tasas:", df_tasas_base.shape)
df_tasas_base.head()

Base para tasas: (12579, 12)


,provincia_id,provincia_nombre,departamento_id,departamento_nombre,anio,delitos_propiedad_hechos,delitos_propiedad_victimas,provincia_key,departamento_key,departamento_id_geo,provincia_geo,departamento_geo
0,2,ciudad autónoma de buenos aires,2001,comuna 1,2000,0,0.0,caba,comuna 1,2007.0,caba,comuna 1
1,2,ciudad autónoma de buenos aires,2001,comuna 1,2001,0,0.0,caba,comuna 1,2007.0,caba,comuna 1
2,2,ciudad autónoma de buenos aires,2001,comuna 1,2002,0,0.0,caba,comuna 1,2007.0,caba,comuna 1
3,2,ciudad autónoma de buenos aires,2001,comuna 1,2003,0,0.0,caba,comuna 1,2007.0,caba,comuna 1
4,2,ciudad autónoma de buenos aires,2001,comuna 1,2004,0,0.0,caba,comuna 1,2007.0,caba,comuna 1


In [25]:
df_tasas_base.isna().sum()

,0
provincia_id,0
provincia_nombre,0
departamento_id,0
departamento_nombre,0
anio,0
delitos_propiedad_hechos,0
delitos_propiedad_victimas,0
provincia_key,0
departamento_key,0
departamento_id_geo,0


# cargando dataset poblacion

In [26]:
# Guardar base final para tasas
df_pob = pd.read_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/raw/poblacion_departamentos_2022.csv")
df_pob.head()

,redcode,count_persona_0,Unnamed: 2
0,2007,221001,NaN
1,2014,160609,NaN
2,2021,193537,NaN
3,2028,227024,NaN
4,2035,192449,NaN


In [27]:

df_pob.columns

Index(['redcode', 'count_persona_0', 'Unnamed: 2'], dtype='object')

In [28]:
# Seleccionar solo las columnas necesarias y renombrarlas para facilitar el merge
df_pob = df_pob[["redcode", "count_persona_0"]].rename(columns={
    "redcode": "departamento_id_geo",
    "count_persona_0": "poblacion_2022"
})

In [29]:
df_pob.head()

,departamento_id_geo,poblacion_2022
0,2007,221001
1,2014,160609
2,2021,193537
3,2028,227024
4,2035,192449


# Merge con poblacion

In [30]:
df_tasas = df_tasas_base.merge(
    df_pob,
    on="departamento_id_geo",
    how="left"
)

In [31]:
df_tasas["poblacion_2022"].isna().sum()

np.int64(0)

In [32]:

df_tasas["tasa_delitos_propiedad_100k"] = (
    df_tasas["delitos_propiedad_hechos"] / df_tasas["poblacion_2022"]
) * 100000

In [33]:
# Mostrar resultado final
df_tasas[[
    "provincia_nombre",
    "departamento_nombre",
    "anio",
    "delitos_propiedad_hechos",
    "poblacion_2022",
    "tasa_delitos_propiedad_100k"
]].head()

,provincia_nombre,departamento_nombre,anio,delitos_propiedad_hechos,poblacion_2022,tasa_delitos_propiedad_100k
0,ciudad autónoma de buenos aires,comuna 1,2000,0,221001,0.0
1,ciudad autónoma de buenos aires,comuna 1,2001,0,221001,0.0
2,ciudad autónoma de buenos aires,comuna 1,2002,0,221001,0.0
3,ciudad autónoma de buenos aires,comuna 1,2003,0,221001,0.0
4,ciudad autónoma de buenos aires,comuna 1,2004,0,221001,0.0


In [34]:
# Calcular tasa por cada 100 habitantes (en porcentaje)
df_tasas["tasa_delitos_propiedad_pct"] = (
    df_tasas["delitos_propiedad_hechos"] / df_tasas["poblacion_2022"]
) * 100

In [35]:
# Mostrar departamentos con mayor tasa por cada 100k habitantes
df_tasas[[
    "provincia_nombre",
    "departamento_nombre",
    "anio",
    "delitos_propiedad_hechos",
    "poblacion_2022",
    "tasa_delitos_propiedad_100k"
]].sort_values("tasa_delitos_propiedad_100k", ascending=False).head(20)

,provincia_nombre,departamento_nombre,anio,delitos_propiedad_hechos,poblacion_2022,tasa_delitos_propiedad_100k
3474,buenos aires,tordillo,2009,1600,2528,63291.139241
2061,buenos aires,laprida,2009,2732,11496,23764.787752
3599,buenos aires,tres lomas,2009,1350,9090,14851.485149
7943,mendoza,capital,2014,16223,123715,13113.203734
7944,mendoza,capital,2015,16038,123715,12963.666492
7938,mendoza,capital,2009,15673,123715,12668.633553
7941,mendoza,capital,2012,15630,123715,12633.876248
7942,mendoza,capital,2013,15053,123715,12167.481712
7940,mendoza,capital,2011,14991,123715,12117.366528
7939,mendoza,capital,2010,14787,123715,11952.471406


# Guardado de datasets

## delitos

In [36]:
cols_real = [
    "provincia_id", "provincia_nombre",
    "departamento_id", "departamento_nombre",
    "anio", "delitos_propiedad_hechos"
]

df_delitos_real = df_delitos[cols_real].copy()

## Tasas

In [37]:
cols_tasas = [
    "provincia_id", "provincia_nombre",
    "departamento_id", "departamento_nombre",
    "anio", "delitos_propiedad_hechos",
    "poblacion_2022",
    "tasa_delitos_propiedad_100k",
    "tasa_delitos_propiedad_pct"
]

df_tasas_final = df_tasas[cols_tasas].copy()

In [38]:
df_delitos_real.to_csv(
    r"C:\Users\Marcos\Ciencia datos e inteligencia artificial\SEXTO SEMESTRE\PP3\Copia_pre_proyecto\data\processed\delitos_propiedad_departamento_anual_real.csv",
    index=False
)

df_tasas_final.to_csv(
    r"C:\Users\Marcos\Ciencia datos e inteligencia artificial\SEXTO SEMESTRE\PP3\Copia_pre_proyecto\data\processed\delitos_propiedad_departamento_anual_tasas.csv",
    index=False
)

# MERGE CON EL RESTO DE LAS VARIABLES

In [39]:
 # Carga de datasets
delitos_propiedad_departamento_anual_real = pd.read_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/processed/delitos_propiedad_departamento_anual_real.csv")
delitos_propiedad_departamento_anual_tasas = pd.read_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/processed/delitos_propiedad_departamento_anual_tasas.csv")
cbt_variacion_anual = pd.read_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/processed/INDEC/cbt_variacion_anual.csv")
empleo_const_anual = pd.read_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/processed/INDEC/empleo_const_anual.csv")
ipc= pd.read_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/processed/INDEC/ipc.csv")
ipim_variacion_anual = pd.read_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/processed/INDEC/ipim_variacion_anual.csv")
salarios_variacion_anual = pd.read_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/processed/INDEC/salarios_variacion_anual.csv")
variacion_accesos_internet_provincias_anual = pd.read_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/processed/INDEC/variacion_accesos_internet_provincias_anual.csv")


In [40]:
print(delitos_propiedad_departamento_anual_real.columns.to_list())
print(delitos_propiedad_departamento_anual_tasas.columns.to_list())
print(cbt_variacion_anual.columns.to_list())
print(empleo_const_anual.columns.to_list())
print(ipc.columns.to_list())
print(ipim_variacion_anual.columns.to_list())
print(salarios_variacion_anual.columns.to_list())
print(variacion_accesos_internet_provincias_anual.columns.to_list())

['provincia_id', 'provincia_nombre', 'departamento_id', 'departamento_nombre', 'anio', 'delitos_propiedad_hechos']
['provincia_id', 'provincia_nombre', 'departamento_id', 'departamento_nombre', 'anio', 'delitos_propiedad_hechos', 'poblacion_2022', 'tasa_delitos_propiedad_100k', 'tasa_delitos_propiedad_pct']
['Unnamed: 0', 'anio', 'variacion_anual_cbt']
['Unnamed: 0', 'anio', 'variacion_empleo_const_pct']
['region', 'anio', 'Indice_IPC']
['anio', 'mes', 'variacion_anual_ipim']
['anio', 'variacion_anual_salarios_pct']
['jurisdiccion', 'anio', 'variacion_acceso_internet_pct']


In [41]:
# Crear diccionario provincia → región . Para agregar la columna de región a los datasets de delitos y poder mergear correctamente con IPC
map_provincia_region = {
    "ciudad autónoma de buenos aires": "GBA",
    "buenos aires": "Pampeana",
    "córdoba": "Pampeana",
    "santa fe": "Pampeana",
    "entre ríos": "Pampeana",
    "la pampa": "Pampeana",

    "mendoza": "Cuyo",
    "san juan": "Cuyo",
    "san luis": "Cuyo",

    "corrientes": "Noreste",
    "chaco": "Noreste",
    "formosa": "Noreste",
    "misiones": "Noreste",

    "catamarca": "Noroeste",
    "tucumán": "Noroeste",
    "salta": "Noroeste",
    "jujuy": "Noroeste",
    "la rioja": "Noroeste",
    "santiago del estero": "Noroeste",

    "neuquén": "Patagonia",
    "río negro": "Patagonia",
    "chubut": "Patagonia",
    "santa cruz": "Patagonia",
    "tierra del fuego, antártida e islas del atlántico sur": "Patagonia"
}

In [42]:
# Agrego columna de región a los datasets de delitos
delitos_propiedad_departamento_anual_real["region"] = delitos_propiedad_departamento_anual_real["provincia_nombre"].map(map_provincia_region)

# Valido que se haya agregado correctamente
delitos_propiedad_departamento_anual_real["region"].isna().sum()

np.int64(0)

In [43]:
# Agrego columna de región a los datasets de delitos
delitos_propiedad_departamento_anual_tasas["region"] = delitos_propiedad_departamento_anual_tasas["provincia_nombre"].map(map_provincia_region)

# Valido que se haya agregado correctamente
delitos_propiedad_departamento_anual_tasas["region"].isna().sum()

np.int64(0)

In [44]:
delitos_propiedad_departamento_anual_tasas.shape

(12579, 10)

In [45]:
# Elimino las columnas unnamed de los dataserts cbt_variacion_anual y empleo_const_anual (que solo fue un id creado al guardar el csv, no es información relevante)
cbt_variacion_anual = cbt_variacion_anual.drop(columns=["Unnamed: 0"], errors="ignore")
empleo_const_anual = empleo_const_anual.drop(columns=["Unnamed: 0"], errors="ignore")

In [46]:
# Inicio el MERGE

df_final = delitos_propiedad_departamento_anual_tasas.copy()



In [47]:
df_final.shape

(12579, 10)

In [48]:
df_final.columns

Index(['provincia_id', 'provincia_nombre', 'departamento_id',
       'departamento_nombre', 'anio', 'delitos_propiedad_hechos',
       'poblacion_2022', 'tasa_delitos_propiedad_100k',
       'tasa_delitos_propiedad_pct', 'region'],
      dtype='object')

In [49]:
# Internet por provincia
df_final = df_final.merge(
    variacion_accesos_internet_provincias_anual,
    left_on=["provincia_nombre", "anio"],
    right_on=["jurisdiccion", "anio"],
    how="left"
)

df_final.columns.to_list()

['provincia_id',
 'provincia_nombre',
 'departamento_id',
 'departamento_nombre',
 'anio',
 'delitos_propiedad_hechos',
 'poblacion_2022',
 'tasa_delitos_propiedad_100k',
 'tasa_delitos_propiedad_pct',
 'region',
 'jurisdiccion',
 'variacion_acceso_internet_pct']

In [50]:
# IPC por region
df_final = df_final.merge(
    ipc,
    on=["region", "anio"],
    how="left"
)
df_final.columns.to_list()

['provincia_id',
 'provincia_nombre',
 'departamento_id',
 'departamento_nombre',
 'anio',
 'delitos_propiedad_hechos',
 'poblacion_2022',
 'tasa_delitos_propiedad_100k',
 'tasa_delitos_propiedad_pct',
 'region',
 'jurisdiccion',
 'variacion_acceso_internet_pct',
 'Indice_IPC']

In [51]:
# Variables macro (solo por año, no por provincia porque no tengo datos provinciales para todas las variables macro)

df_final = df_final.merge(salarios_variacion_anual, on="anio", how="left")
df_final = df_final.merge(cbt_variacion_anual, on="anio", how="left")
df_final = df_final.merge(empleo_const_anual, on="anio", how="left")
df_final = df_final.merge(ipim_variacion_anual, on="anio", how="left")
df_final.columns.to_list()

['provincia_id',
 'provincia_nombre',
 'departamento_id',
 'departamento_nombre',
 'anio',
 'delitos_propiedad_hechos',
 'poblacion_2022',
 'tasa_delitos_propiedad_100k',
 'tasa_delitos_propiedad_pct',
 'region',
 'jurisdiccion',
 'variacion_acceso_internet_pct',
 'Indice_IPC',
 'variacion_anual_salarios_pct',
 'variacion_anual_cbt',
 'variacion_empleo_const_pct',
 'mes',
 'variacion_anual_ipim']

In [52]:
# Elimino columna "jurisdiccion" con informacion duplicada de "provincia_nombre"
df_final = df_final.drop(columns=["jurisdiccion"], errors="ignore")

In [53]:
print(df_final.shape)
print(df_final.columns.tolist())
df_final.head()

(12579, 17)
['provincia_id', 'provincia_nombre', 'departamento_id', 'departamento_nombre', 'anio', 'delitos_propiedad_hechos', 'poblacion_2022', 'tasa_delitos_propiedad_100k', 'tasa_delitos_propiedad_pct', 'region', 'variacion_acceso_internet_pct', 'Indice_IPC', 'variacion_anual_salarios_pct', 'variacion_anual_cbt', 'variacion_empleo_const_pct', 'mes', 'variacion_anual_ipim']


,provincia_id,provincia_nombre,departamento_id,departamento_nombre,anio,delitos_propiedad_hechos,poblacion_2022,tasa_delitos_propiedad_100k,tasa_delitos_propiedad_pct,region,variacion_acceso_internet_pct,Indice_IPC,variacion_anual_salarios_pct,variacion_anual_cbt,variacion_empleo_const_pct,mes,variacion_anual_ipim
0,2,ciudad autónoma de buenos aires,2001,comuna 1,2000,0,221001,0.0,0.0,GBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,ciudad autónoma de buenos aires,2001,comuna 1,2001,0,221001,0.0,0.0,GBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,ciudad autónoma de buenos aires,2001,comuna 1,2002,0,221001,0.0,0.0,GBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,ciudad autónoma de buenos aires,2001,comuna 1,2003,0,221001,0.0,0.0,GBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,ciudad autónoma de buenos aires,2001,comuna 1,2004,0,221001,0.0,0.0,GBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
df_final.isna().sum().sort_values(ascending=False)

,0
Indice_IPC,12579
variacion_acceso_internet_pct,12579
variacion_anual_cbt,8507
variacion_anual_salarios_pct,8507
variacion_empleo_const_pct,8006
variacion_anual_ipim,8006
mes,7505
provincia_id,0
provincia_nombre,0
tasa_delitos_propiedad_pct,0


In [55]:
df_final.duplicated(
    subset=["provincia_nombre", "departamento_nombre", "anio"]
).sum()

np.int64(0)

In [56]:
df_final.describe().T

,count,mean,std,min,25%,50%,75%,max
provincia_id,12579.0,36.083472,29.401261,2.000000,6.000000,26.000000,62.000000,9.400000e+01
departamento_id,12579.0,36252.861197,29298.990980,2001.000000,6728.000000,26077.000000,62070.000000,9.401400e+04
anio,12579.0,2012.061929,7.234970,2000.000000,2006.000000,2012.000000,2018.000000,2.024000e+03
delitos_propiedad_hechos,12579.0,1334.536052,4001.395198,0.000000,78.000000,246.000000,923.000000,9.872900e+04
poblacion_2022,12579.0,89480.078862,166247.325112,413.000000,14092.000000,33951.000000,93464.000000,1.837168e+06
tasa_delitos_propiedad_100k,12579.0,1161.577339,1143.540686,0.000000,556.195623,940.929440,1522.381946,6.329114e+04
tasa_delitos_propiedad_pct,12579.0,1.161577,1.143541,0.000000,0.556196,0.940929,1.522382,6.329114e+01
variacion_acceso_internet_pct,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Indice_IPC,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
variacion_anual_salarios_pct,4072.0,72.531758,48.832220,27.470000,32.990000,53.360000,145.470000,1.526800e+02


Verificar por que no matchearon correctamente IPC y accesos internet

In [57]:
print(df_final["provincia_nombre"].unique())
print(variacion_accesos_internet_provincias_anual["jurisdiccion"].unique())

['ciudad autónoma de buenos aires' 'buenos aires' 'catamarca' 'córdoba'
 'corrientes' 'chaco' 'chubut' 'entre ríos' 'formosa' 'jujuy' 'la pampa'
 'la rioja' 'mendoza' 'misiones' 'neuquén' 'río negro' 'salta' 'san juan'
 'san luis' 'santa cruz' 'santa fe' 'santiago del estero' 'tucumán'
 'tierra del fuego, antártida e islas del atlántico sur']
['CABA y provincia de Buenos Aires' 'Catamarca' 'Chaco' 'Chubut'
 'Corrientes' 'Córdoba' 'Entre Ríos' 'Formosa' 'Jujuy' 'La Pampa'
 'La Rioja' 'Mendoza' 'Misiones' 'Neuquén' 'Río Negro' 'Salta' 'San Juan'
 'San Luis' 'Santa Cruz' 'Santa Fe' 'Santiago del Estero'
 'Tierra del Fuego, Antártida e Islas del Atlántico Sur' 'Total del país'
 'Tucumán']


In [58]:
print(df_final["region"].unique())
print(ipc["region"].unique())

['GBA' 'Pampeana' 'Noroeste' 'Noreste' 'Patagonia' 'Cuyo']
['cuyo' 'gba' 'nea' 'noa' 'pampeana' 'patagonia']


In [59]:
# Arreglar merge con variable de acceso a internet
df_internet = variacion_accesos_internet_provincias_anual.copy()

# Normalizar texto
df_internet["jurisdiccion"] = df_internet["jurisdiccion"].str.lower()

# Separar CABA + Buenos Aires
df_ba = df_internet[
    df_internet["jurisdiccion"] == "caba y provincia de buenos aires"
].copy()

df_ba_caba = df_ba.copy()
df_ba_caba["jurisdiccion"] = "ciudad autónoma de buenos aires"

df_ba_pba = df_ba.copy()
df_ba_pba["jurisdiccion"] = "buenos aires"

# Eliminar original y reemplazar
df_internet = df_internet[
    df_internet["jurisdiccion"] != "caba y provincia de buenos aires"
]

df_internet = pd.concat([df_internet, df_ba_caba, df_ba_pba], ignore_index=True)

In [60]:
# Rehacer el merge
df_final = df_final.drop(columns=["variacion_acceso_internet_pct"], errors="ignore")

df_final = df_final.merge(
    df_internet,
    left_on=["provincia_nombre", "anio"],
    right_on=["jurisdiccion", "anio"],
    how="left"
)

In [61]:
# Arreglar variable IPC (reemplazar regiones por provincias para hacer merge correcto)
map_region = {
    "gba": "GBA",
    "pampeana": "Pampeana",
    "cuyo": "Cuyo",
    "nea": "Noreste",
    "noa": "Noroeste",
    "patagonia": "Patagonia"
}

# Aplicar mapeo a la columna de región en el dataset de IPC
ipc["region"] = ipc["region"].str.lower().map(map_region)

In [62]:
# Rehacer merge
df_final = df_final.drop(columns=["Indice_IPC"], errors="ignore")

df_final = df_final.merge(
    ipc,
    on=["region", "anio"],
    how="left"
)

In [64]:
# Validar
df_final.isna().sum().sort_values(ascending=False)

,0
variacion_acceso_internet_pct,8507
variacion_anual_cbt,8507
variacion_anual_salarios_pct,8507
Indice_IPC,8006
variacion_anual_ipim,8006
variacion_empleo_const_pct,8006
jurisdiccion,8006
mes,7505
provincia_nombre,0
provincia_id,0


In [65]:
df_final.groupby("anio").size()

,0
anio,
2000,500
2001,500
2002,500
2003,500
2004,500
2005,500
2006,500
2007,500
2008,500


In [66]:
# Filtro dataset desde año 2017 en adelante (porque antes hay muchos datos faltantes en las variables macro)
df_modelo = df_final[df_final["anio"] >= 2017].copy()
print(df_modelo.shape)
df_modelo.isna().sum().sort_values(ascending=False)


(4072, 18)


,0
provincia_id,0
provincia_nombre,0
departamento_id,0
departamento_nombre,0
anio,0
delitos_propiedad_hechos,0
poblacion_2022,0
tasa_delitos_propiedad_100k,0
tasa_delitos_propiedad_pct,0
region,0


In [67]:
# Elimino columnas auxiliares innecesarias con datos faltantes
df_modelo = df_modelo.drop(columns=["jurisdiccion_y", "jurisdiccion_x","variacion_acceso_internet_pct_y", "variacion_acceso_internet_pct_x"], errors="ignore")
print(df_modelo.shape)
df_modelo.isna().sum().sort_values(ascending=False)

(4072, 18)


,0
provincia_id,0
provincia_nombre,0
departamento_id,0
departamento_nombre,0
anio,0
delitos_propiedad_hechos,0
poblacion_2022,0
tasa_delitos_propiedad_100k,0
tasa_delitos_propiedad_pct,0
region,0


In [68]:
# Guardar dataset final para modelado
df_modelo.to_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/final/df_modelo.csv", index=False)
df_modelo.shape

(4072, 18)

In [69]:
# Creo un dataset copia del dataset final, solo del año 2024

df_mapa_2024 = df_modelo [df_modelo["anio"] == 2024].copy()

df_mapa_2024 = df_mapa_2024[[
    "provincia_id",
    "provincia_nombre",
    "departamento_id",
    "departamento_nombre",
    "anio",
    "delitos_propiedad_hechos",
    "tasa_delitos_propiedad_100k",
    "tasa_delitos_propiedad_pct"
]].copy()

In [70]:
# Guardo dataset
df_mapa_2024.to_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/final/dataset_mapa_2024.csv", index=False)

 # Dataset proyecciones de poblacion por año + datos superficie departamento

In [71]:


path_pob = r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/raw/Proyecciones_departamento_2010-2025.csv"

df_proy_pob = pd.read_csv(path_pob)
df_proy_pob.head()

,Código de provincia,Provincia,Código de departamento,Departamento,Año 2010,Año 2011,Año 2012,Año 2013,Año 2014,Año 2015,...,Código de provincia.1,Nombre de provincia,Código de departamento.1,Nombre de departamento,Población total,Total de hogares,Latitud del centroide,Longitud del centroide,Superficie en km2,Geometría en GeoJSON
0,2,CABA,2001,Comuna 1,243946,245308,246689,248069,249433,250770,...,2,Ciudad Autónoma de Buenos Aires,2001,Comuna 1,205886,84468,-34.606365,-58.371709,17.774344,"{""type"":""Polygon"",""coordinates"":[[[-58.386301,..."
1,2,CABA,2002,Comuna 2,150714,150573,150428,150278,150130,149985,...,2,Ciudad Autónoma de Buenos Aires,2002,Comuna 2,157932,73156,-34.585876,-58.394997,6.276148,"{""type"":""Polygon"",""coordinates"":[[[-58.381109,..."
2,2,CABA,2003,Comuna 3,191323,191536,191750,191963,192171,192375,...,2,Ciudad Autónoma de Buenos Aires,2003,Comuna 3,187537,80489,-34.613826,-58.402686,6.386040,"{""type"":""Polygon"",""coordinates"":[[[-58.411917,..."
3,2,CABA,2004,Comuna 4,234933,235497,236071,236646,237214,237769,...,2,Ciudad Autónoma de Buenos Aires,2004,Comuna 4,218245,76455,-34.642292,-58.388860,21.694410,"{""type"":""MultiPolygon"",""coordinates"":[[[[-58.3..."
4,2,CABA,2005,Comuna 5,185301,185544,185789,186034,186276,186512,...,2,Ciudad Autónoma de Buenos Aires,2005,Comuna 5,179005,76846,-34.617353,-58.420607,6.659799,"{""type"":""Polygon"",""coordinates"":[[[-58.417264,..."


In [72]:
df_proy_pob.columns.tolist()


['Código de provincia',
 'Provincia',
 'Código de departamento',
 'Departamento',
 'Año 2010',
 'Año 2011',
 'Año 2012',
 'Año 2013',
 'Año 2014',
 'Año 2015',
 'Año 2016',
 'Año 2017',
 'Año 2018',
 'Año 2019',
 'Año 2020',
 'Año 2021',
 'Año 2022',
 'Año 2023',
 'Año 2024',
 'Año 2025',
 'Código de provincia.1',
 'Nombre de provincia',
 'Código de departamento.1',
 'Nombre de departamento',
 'Población total',
 'Total de hogares',
 'Latitud del centroide',
 'Longitud del centroide',
 'Superficie en km2',
 'Geometría en GeoJSON']

In [73]:
# Pasar dataset a formato largo
# 1. Identificar columnas de años
cols_anios = [col for col in df_proy_pob.columns if "Año" in col]

# 2. Pasar a formato largo
df_pob_long = df_proy_pob.melt(
    id_vars=[
        "Provincia",
        "Departamento",
        "Código de provincia",
        "Código de departamento",
        "Superficie en km2"
    ],
    value_vars=cols_anios,
    var_name="anio",
    value_name="poblacion"
)

# 3. Limpiar columna año
df_pob_long["anio"] = df_pob_long["anio"].str.replace("Año ", "").astype(int)

# 4. Ver resultado
df_pob_long.head()


,Provincia,Departamento,Código de provincia,Código de departamento,Superficie en km2,anio,poblacion
0,CABA,Comuna 1,2,2001,17.774344,2010,243946.0
1,CABA,Comuna 2,2,2002,6.276148,2010,150714.0
2,CABA,Comuna 3,2,2003,6.386040,2010,191323.0
3,CABA,Comuna 4,2,2004,21.694410,2010,234933.0
4,CABA,Comuna 5,2,2005,6.659799,2010,185301.0


In [74]:
df_pob_long.tail()

,Provincia,Departamento,Código de provincia,Código de departamento,Superficie en km2,anio,poblacion
8395,Tucumán,Tafí Viejo,90,90105,1139.078712,2025,162025.0
8396,Tucumán,Trancas,90,90112,3068.140292,2025,23033.0
8397,Tucumán,Yerba Buena,90,90119,149.164148,2025,105914.0
8398,Tierra del Fuego,Río Grande,94,94007,11767.490783,2025,107928.0
8399,Tierra del Fuego,Ushuaia,94,94014,8910.185239,2025,86678.0


In [75]:
# Cargar df_mapa_final
df_mapa_final = pd.read_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/final/dataset_mapa_final.csv")
df_mapa_final.shape

(4072, 20)

In [77]:
import unicodedata
import re

def normalizar_texto(s):
    if pd.isna(s):
        return s
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8")
    s = re.sub(r"\s+", " ", s)
    return s

df_pob_long["provincia_key"] = df_pob_long["Provincia"].apply(normalizar_texto)
df_pob_long["departamento_key"] = df_pob_long["Departamento"].apply(normalizar_texto)

df_mapa_final["provincia_key"] = df_mapa_final["provincia_nombre"].apply(normalizar_texto)
df_mapa_final["departamento_key"] = df_mapa_final["departamento_nombre"].apply(normalizar_texto)

df_mapa_final.columns.tolist()

['provincia_id',
 'provincia_nombre',
 'departamento_id',
 'departamento_nombre',
 'anio',
 'delitos_propiedad_hechos',
 'poblacion_2022',
 'tasa_delitos_propiedad_100k',
 'tasa_delitos_propiedad_pct',
 'region',
 'variacion_anual_salarios_pct',
 'variacion_anual_cbt',
 'variacion_empleo_const_pct',
 'mes',
 'variacion_anual_ipim',
 'jurisdiccion',
 'variacion_acceso_internet_pct',
 'Indice_IPC',
 'lat',
 'lon',
 'provincia_key',
 'departamento_key']

In [78]:
# Merge
df_mapa_sup_final = df_mapa_final.merge(
    df_pob_long[[
        "provincia_key",
        "departamento_key",
        "anio",
        "poblacion",
        "Superficie en km2"
    ]],
    on=["provincia_key", "departamento_key", "anio"],
    how="left"
)

In [79]:
df_mapa_sup_final[["provincia_nombre", "departamento_nombre", "anio", "poblacion", "Superficie en km2"]].head()

,provincia_nombre,departamento_nombre,anio,poblacion,Superficie en km2
0,ciudad autónoma de buenos aires,comuna 1,2017,NaN,NaN
1,ciudad autónoma de buenos aires,comuna 1,2018,NaN,NaN
2,ciudad autónoma de buenos aires,comuna 1,2019,NaN,NaN
3,ciudad autónoma de buenos aires,comuna 1,2020,NaN,NaN
4,ciudad autónoma de buenos aires,comuna 1,2021,NaN,NaN


In [80]:
# Ver ejemplos de claves en ambos datasets

print("df_mapa_final:")
print(df_mapa_final[["provincia_nombre", "departamento_nombre", "provincia_key", "departamento_key"]].drop_duplicates().head(10))

print("\ndf_pob_long:")
print(df_pob_long[["Provincia", "Departamento", "provincia_key", "departamento_key"]].drop_duplicates().head(10))

df_mapa_final:
                   provincia_nombre departamento_nombre  \
0   ciudad autónoma de buenos aires            comuna 1   
8   ciudad autónoma de buenos aires            comuna 2   
16  ciudad autónoma de buenos aires            comuna 3   
24  ciudad autónoma de buenos aires            comuna 4   
32  ciudad autónoma de buenos aires            comuna 5   
40  ciudad autónoma de buenos aires            comuna 6   
48  ciudad autónoma de buenos aires            comuna 7   
56  ciudad autónoma de buenos aires            comuna 8   
64  ciudad autónoma de buenos aires            comuna 9   
72  ciudad autónoma de buenos aires           comuna 10   

                      provincia_key departamento_key  
0   ciudad autonoma de buenos aires         comuna 1  
8   ciudad autonoma de buenos aires         comuna 2  
16  ciudad autonoma de buenos aires         comuna 3  
24  ciudad autonoma de buenos aires         comuna 4  
32  ciudad autonoma de buenos aires         comuna 5  
40  c

In [81]:
# Normalizar provincia en df_pob_long
df_pob_long["provincia_key"] = df_pob_long["provincia_key"].replace({
    "caba": "ciudad autonoma de buenos aires"
})

In [82]:
df_mapa_sup_final = df_mapa_final.merge(
    df_pob_long[[
        "provincia_key",
        "departamento_key",
        "anio",
        "poblacion",
        "Superficie en km2"
    ]],
    on=["provincia_key", "departamento_key", "anio"],
    how="left",
    indicator=True
)

df_mapa_sup_final["_merge"].value_counts()

,count
_merge,
both,3979
left_only,93
right_only,0


In [83]:
faltantes_merge = df_mapa_sup_final.loc[
    df_mapa_sup_final["_merge"] == "left_only",
    ["provincia_nombre", "departamento_nombre", "anio", "provincia_key", "departamento_key"]
].drop_duplicates().sort_values(["provincia_nombre", "departamento_nombre", "anio"])

faltantes_merge.head(50)

,provincia_nombre,departamento_nombre,anio,provincia_key,departamento_key
322,buenos aires,coronel de marina l. rosales,2017,buenos aires,coronel de marina l. rosales
323,buenos aires,coronel de marina l. rosales,2018,buenos aires,coronel de marina l. rosales
324,buenos aires,coronel de marina l. rosales,2019,buenos aires,coronel de marina l. rosales
325,buenos aires,coronel de marina l. rosales,2020,buenos aires,coronel de marina l. rosales
326,buenos aires,coronel de marina l. rosales,2021,buenos aires,coronel de marina l. rosales
327,buenos aires,coronel de marina l. rosales,2022,buenos aires,coronel de marina l. rosales
328,buenos aires,coronel de marina l. rosales,2023,buenos aires,coronel de marina l. rosales
329,buenos aires,coronel de marina l. rosales,2024,buenos aires,coronel de marina l. rosales
682,buenos aires,lezama,2017,buenos aires,lezama
683,buenos aires,lezama,2018,buenos aires,lezama


In [84]:
faltantes_merge[["provincia_nombre", "departamento_nombre"]].drop_duplicates().sort_values(
    ["provincia_nombre", "departamento_nombre"]
)

,provincia_nombre,departamento_nombre
322,buenos aires,coronel de marina l. rosales
682,buenos aires,lezama
1864,chaco,o' higgins
1872,chaco,presidencia de la plaza
1336,córdoba,cordoba capital
2415,la pampa,chicalcó
2438,la pampa,quemú ouemú
2584,mendoza,mendoza capital
2808,misiones,libertador grl. san martín
3152,salta,grl. josé de san martín


In [85]:
# Correcciones manuales de nombres para mejorar el match
reemplazos_provincias_mapa = {
    "tierra del fuego, antartida e islas del atlantico sur": "tierra del fuego"
}

df_mapa_final["provincia_key"] = df_mapa_final["provincia_key"].replace(reemplazos_provincias_mapa)

In [86]:
# Mergear nuevamente
df_mapa_sup_final = df_mapa_final.merge(
    df_pob_long[[
        "provincia_key",
        "departamento_key",
        "anio",
        "poblacion",
        "Superficie en km2"
    ]],
    on=["provincia_key", "departamento_key", "anio"],
    how="left",
    indicator=True
)

df_mapa_sup_final["_merge"].value_counts()

,count
_merge,
both,3995
left_only,77
right_only,0


In [87]:
# Verificar nuevamente los merge faltantes
faltantes_merge = df_mapa_sup_final.loc[
    df_mapa_sup_final["_merge"] == "left_only",
    ["provincia_nombre", "departamento_nombre", "anio", "provincia_key", "departamento_key"]
].drop_duplicates().sort_values(["provincia_nombre", "departamento_nombre", "anio"])

faltantes_merge[["provincia_nombre", "departamento_nombre"]].drop_duplicates().sort_values(
    ["provincia_nombre", "departamento_nombre"]
)

,provincia_nombre,departamento_nombre
322,buenos aires,coronel de marina l. rosales
682,buenos aires,lezama
1864,chaco,o' higgins
1872,chaco,presidencia de la plaza
1336,córdoba,cordoba capital
2415,la pampa,chicalcó
2438,la pampa,quemú ouemú
2584,mendoza,mendoza capital
2808,misiones,libertador grl. san martín
3152,salta,grl. josé de san martín


In [88]:
faltantes = df_mapa_sup_final[df_mapa_sup_final["_merge"] == "left_only"]

faltantes.head(50)

,provincia_id,provincia_nombre,departamento_id,departamento_nombre,anio,delitos_propiedad_hechos,poblacion_2022,tasa_delitos_propiedad_100k,tasa_delitos_propiedad_pct,region,...,jurisdiccion,variacion_acceso_internet_pct,Indice_IPC,lat,lon,provincia_key,departamento_key,poblacion,Superficie en km2,_merge
322,6,buenos aires,6182,coronel de marina l. rosales,2017,259,65823,393.479483,0.393479,Pampeana,...,buenos aires,4.281205,112.744967,-38.849077,-61.835584,buenos aires,coronel de marina l. rosales,NaN,NaN,left_only
323,6,buenos aires,6182,coronel de marina l. rosales,2018,443,65823,673.017031,0.673017,Pampeana,...,buenos aires,-0.084572,151.744600,-38.849077,-61.835584,buenos aires,coronel de marina l. rosales,NaN,NaN,left_only
324,6,buenos aires,6182,coronel de marina l. rosales,2019,733,65823,1113.592513,1.113593,Pampeana,...,buenos aires,3.259377,232.926758,-38.849077,-61.835584,buenos aires,coronel de marina l. rosales,NaN,NaN,left_only
325,6,buenos aires,6182,coronel de marina l. rosales,2020,716,65823,1087.765675,1.087766,Pampeana,...,buenos aires,-0.906055,333.645717,-38.849077,-61.835584,buenos aires,coronel de marina l. rosales,NaN,NaN,left_only
326,6,buenos aires,6182,coronel de marina l. rosales,2021,726,65823,1102.957933,1.102958,Pampeana,...,buenos aires,5.042552,499.073575,-38.849077,-61.835584,buenos aires,coronel de marina l. rosales,NaN,NaN,left_only
327,6,buenos aires,6182,coronel de marina l. rosales,2022,875,65823,1329.322577,1.329323,Pampeana,...,buenos aires,9.714659,855.284483,-38.849077,-61.835584,buenos aires,coronel de marina l. rosales,NaN,NaN,left_only
328,6,buenos aires,6182,coronel de marina l. rosales,2023,1135,65823,1724.321286,1.724321,Pampeana,...,buenos aires,7.971208,1996.292658,-38.849077,-61.835584,buenos aires,coronel de marina l. rosales,NaN,NaN,left_only
329,6,buenos aires,6182,coronel de marina l. rosales,2024,913,65823,1387.053158,1.387053,Pampeana,...,buenos aires,2.207989,6356.215883,-38.849077,-61.835584,buenos aires,coronel de marina l. rosales,NaN,NaN,left_only
682,6,buenos aires,6466,lezama,2017,33,6170,534.846029,0.534846,Pampeana,...,buenos aires,4.281205,112.744967,-35.849208,-57.894844,buenos aires,lezama,NaN,NaN,left_only
683,6,buenos aires,6466,lezama,2018,43,6170,696.920583,0.696921,Pampeana,...,buenos aires,-0.084572,151.744600,-35.849208,-57.894844,buenos aires,lezama,NaN,NaN,left_only


In [89]:
# Buscar en df_pob_long coincidencias de nombres de los departamentos faltantes
# Buscar aproximaciones más amplias
busquedas = [
    "rosales",
    "lezama",
    "higgins",
    "plaza",
    "chical",
    "alberdi",
    "capital",
    "san mart",
    "quemu"
]

print("df_pob_long:")
for b in busquedas:
    print(f"\n--- Buscando: {b} ---")
    display(
        df_pob_long[
            df_pob_long["departamento_key"].str.contains(b, na=False)
        ][["Provincia", "Departamento", "provincia_key", "departamento_key"]].drop_duplicates()
    )

print("df_mapa_final:")
for b in busquedas:
    print(f"\n--- Buscando: {b} ---")
    display(
        df_mapa_final[
            df_mapa_final["departamento_key"].str.contains(b, na=False)
        ][["provincia_nombre", "departamento_nombre", "provincia_key", "departamento_key"]].drop_duplicates()
    )

df_pob_long:

--- Buscando: rosales ---


,Provincia,Departamento,provincia_key,departamento_key
45,Buenos Aires,Coronel Leonardo Rosales,buenos aires,coronel leonardo rosales



--- Buscando: lezama ---


,Provincia,Departamento,provincia_key,departamento_key



--- Buscando: higgins ---


...

,provincia_nombre,departamento_nombre,provincia_key,departamento_key
1248,catamarca,capital,catamarca,capital
1336,córdoba,cordoba capital,cordoba,cordoba capital
1552,corrientes,capital,corrientes,capital
2397,la pampa,capital,la pampa,capital
2464,la rioja,capital,la rioja,capital
2584,mendoza,mendoza capital,mendoza,mendoza capital
2752,misiones,capital,misiones,capital
3120,salta,capital,salta,capital
3304,san juan,capital,san juan,capital
3616,santa fe,la capital,santa fe,la capital



--- Buscando: san mart ---


,provincia_nombre,departamento_nombre,provincia_key,departamento_key
554,buenos aires,general san martín,buenos aires,general san martin
1368,córdoba,general san martín,cordoba,general san martin
1696,corrientes,san martín,corrientes,san martin
1832,chaco,libertador general san martín,chaco,libertador general san martin
2536,la rioja,general san martín,la rioja,general san martin
2688,mendoza,san martín,mendoza,san martin
2808,misiones,libertador grl. san martín,misiones,libertador grl. san martin
3152,salta,grl. josé de san martín,salta,grl. jose de san martin
3376,san juan,san martín,san juan,san martin
3488,san luis,libertador general san martín,san luis,libertador general san martin



--- Buscando: quemu ---


,provincia_nombre,departamento_nombre,provincia_key,departamento_key
2438,la pampa,quemú ouemú,la pampa,quemu ouemu


In [90]:
df_pob_long[
    df_pob_long["Departamento"].str.contains("Lezama", case=False, na=False)
][["Provincia", "Departamento", "departamento_key"]].drop_duplicates()

,Provincia,Departamento,departamento_key


In [91]:
print(
    df_pob_long[
        df_pob_long["provincia_key"] == "buenos aires"
    ][["Departamento", "departamento_key"]]
    .drop_duplicates()
    .sort_values("Departamento")
    .to_string(index=False)
)

            Departamento         departamento_key
              25 de Mayo               25 de mayo
              9 de Julio               9 de julio
           Adolfo Alsina            adolfo alsina
  Adolfo Gonzales Chaves   adolfo gonzales chaves
                 Alberti                  alberti
         Almirante Brown          almirante brown
               Arrecifes                arrecifes
              Avellaneda               avellaneda
                Ayacucho                 ayacucho
                    Azul                     azul
            Bahía Blanca             bahia blanca
                Balcarce                 balcarce
                Baradero                 baradero
           Benito Juárez            benito juarez
             Berazategui              berazategui
                 Berisso                  berisso
                 Bolívar                  bolivar
                 Bragado                  bragado
                Brandsen                 brandsen


In [92]:
reemplazos_finales = {
    "coronel de marina l. rosales": "coronel leonardo rosales",
    "o' higgins": "o'higgins",
    "presidencia de la plaza": "presidente de la plaza",
    "chicalco": "chical co",
    "juan bautista alberdi": "juan b. alberdi",
    "cordoba capital":"capital",
    "mendoza capital":"capital",
    "libertador grl. san martin":"libertador general san martin",
    "grl. jose de san martin":"general jose de san martin",
    "quemu ouemu":"quemu quemu"
}

df_mapa_final["departamento_key"] = df_mapa_final["departamento_key"].replace(reemplazos_finales)

In [93]:
# De nuevo el merge...
df_mapa_sup_final = df_mapa_final.merge(
    df_pob_long[[
        "provincia_key",
        "departamento_key",
        "anio",
        "poblacion",
        "Superficie en km2"
    ]],
    on=["provincia_key", "departamento_key", "anio"],
    how="left",
    indicator=True
)

df_mapa_sup_final["_merge"].value_counts()

,count
_merge,
both,4064
left_only,8
right_only,0


In [94]:

print(df_mapa_sup_final.columns)
df_mapa_sup_final.head()

Index(['provincia_id', 'provincia_nombre', 'departamento_id',
       'departamento_nombre', 'anio', 'delitos_propiedad_hechos',
       'poblacion_2022', 'tasa_delitos_propiedad_100k',
       'tasa_delitos_propiedad_pct', 'region', 'variacion_anual_salarios_pct',
       'variacion_anual_cbt', 'variacion_empleo_const_pct', 'mes',
       'variacion_anual_ipim', 'jurisdiccion', 'variacion_acceso_internet_pct',
       'Indice_IPC', 'lat', 'lon', 'provincia_key', 'departamento_key',
       'poblacion', 'Superficie en km2', '_merge'],
      dtype='object')


,provincia_id,provincia_nombre,departamento_id,departamento_nombre,anio,delitos_propiedad_hechos,poblacion_2022,tasa_delitos_propiedad_100k,tasa_delitos_propiedad_pct,region,...,jurisdiccion,variacion_acceso_internet_pct,Indice_IPC,lat,lon,provincia_key,departamento_key,poblacion,Superficie en km2,_merge
0,2,ciudad autónoma de buenos aires,2001,comuna 1,2017,17609,221001,7967.837250,7.967837,GBA,...,ciudad autónoma de buenos aires,4.281205,112.985075,-34.606444,-58.371512,ciudad autonoma de buenos aires,comuna 1,253271.0,17.774344,both
1,2,ciudad autónoma de buenos aires,2001,comuna 1,2018,18374,221001,8313.989529,8.313990,GBA,...,ciudad autónoma de buenos aires,-0.084572,151.569817,-34.606444,-58.371512,ciudad autonoma de buenos aires,comuna 1,254408.0,17.774344,both
2,2,ciudad autónoma de buenos aires,2001,comuna 1,2019,21721,221001,9828.462315,9.828462,GBA,...,ciudad autónoma de buenos aires,3.259377,231.605092,-34.606444,-58.371512,ciudad autonoma de buenos aires,comuna 1,255457.0,17.774344,both
3,2,ciudad autónoma de buenos aires,2001,comuna 1,2020,9773,221001,4422.151936,4.422152,GBA,...,ciudad autónoma de buenos aires,-0.906055,325.364358,-34.606444,-58.371512,ciudad autonoma de buenos aires,comuna 1,256405.0,17.774344,both
4,2,ciudad autónoma de buenos aires,2001,comuna 1,2021,13012,221001,5887.756164,5.887756,GBA,...,ciudad autónoma de buenos aires,5.042552,478.615058,-34.606444,-58.371512,ciudad autonoma de buenos aires,comuna 1,257235.0,17.774344,both


### Se detectaron inconsistencias menores entre fuentes territoriales (SNIC vs estimaciones poblacionales), resultando en un pequeño porcentaje de observaciones sin correspondencia. Estos casos fueron conservados en el dataset, identificados mediante una variable de control, y no afectan significativamente el análisis global.

In [95]:
# Etiquetar casos faltantes
df_mapa_sup_final["flag_poblacion_faltante"] = df_mapa_sup_final["poblacion"].isna()
df_mapa_sup_final["flag_poblacion_faltante"].sum()

np.int64(8)

### Crear la nueva tasa de delitos con las proyecciones de poblacion

In [96]:
df_mapa_sup_final["tasa_delitos_propiedad_100k_v2"] = (
    df_mapa_sup_final["delitos_propiedad_hechos"] / df_mapa_sup_final["poblacion"]
) * 100000

### Calcular la densidad de la poblacion en cada departamento

In [97]:
df_mapa_sup_final["densidad_poblacion"] = (
    df_mapa_sup_final["poblacion"] / df_mapa_sup_final["Superficie en km2"]
)

# Creación de varibles Lag

In [98]:
import numpy as np

# 0) Ordenar panel por provincia, departamento y año para asegurar que los lags estén correctos
df_mapa_sup_final = df_mapa_sup_final.sort_values(
    ["provincia_key", "departamento_key", "anio"]
)

# 1) Lag del target
df_mapa_sup_final["tasa_lag1"] = df_mapa_sup_final.groupby(
    ["provincia_key", "departamento_key"]
)["tasa_delitos_propiedad_100k_v2"].shift(1)

# 2) Lags de variables externas
df_mapa_sup_final["Indice_IPC_lag1"] = df_mapa_sup_final.groupby(
    ["provincia_key", "departamento_key"]
)["Indice_IPC"].shift(1)

df_mapa_sup_final["variacion_anual_cbt_lag1"] = df_mapa_sup_final.groupby(
    ["provincia_key", "departamento_key"]
)["variacion_anual_cbt"].shift(1)

df_mapa_sup_final["variacion_anual_salarios_lag1"] = df_mapa_sup_final.groupby(
    ["provincia_key", "departamento_key"]
)["variacion_anual_salarios_pct"].shift(1)

# 3) Crecimiento interanual del target
df_mapa_sup_final["tasa_yoy"] = df_mapa_sup_final.groupby(
    ["provincia_key", "departamento_key"]
)["tasa_delitos_propiedad_100k_v2"].pct_change(fill_method=None)

# 4) Log de densidad
df_mapa_sup_final["log_densidad"] = np.log(
    df_mapa_sup_final["densidad_poblacion"] + 1
)

In [99]:
# Guardar el dataset
df_mapa_sup_final.to_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/final/dataset_mapa_sup_final.csv", index=False)

In [ ]:
df_mapa_sup_final.columns.tolist()

['provincia_id',
 'provincia_nombre',
 'departamento_id',
 'departamento_nombre',
 'anio',
 'delitos_propiedad_hechos',
 'poblacion_2022',
 'tasa_delitos_propiedad_100k',
 'tasa_delitos_propiedad_pct',
 'region',
 'variacion_anual_salarios_pct',
 'variacion_anual_cbt',
 'variacion_empleo_const_pct',
 'mes',
 'variacion_anual_ipim',
 'jurisdiccion',
 'variacion_acceso_internet_pct',
 'Indice_IPC',
 'lat',
 'lon',
 'provincia_key',
 'departamento_key',
 'poblacion',
 'Superficie en km2',
 '_merge',
 'flag_poblacion_faltante',
 'tasa_delitos_propiedad_100k_v2',
 'densidad_poblacion',
 'tasa_lag1',
 'Indice_IPC_lag1',
 'variacion_anual_cbt_lag1',
 'variacion_anual_salarios_lag1',
 'tasa_yoy',
 'log_densidad']

In [100]:
print(df_mapa_sup_final.columns.tolist())
df_mapa_sup_final.shape

['provincia_id', 'provincia_nombre', 'departamento_id', 'departamento_nombre', 'anio', 'delitos_propiedad_hechos', 'poblacion_2022', 'tasa_delitos_propiedad_100k', 'tasa_delitos_propiedad_pct', 'region', 'variacion_anual_salarios_pct', 'variacion_anual_cbt', 'variacion_empleo_const_pct', 'mes', 'variacion_anual_ipim', 'jurisdiccion', 'variacion_acceso_internet_pct', 'Indice_IPC', 'lat', 'lon', 'provincia_key', 'departamento_key', 'poblacion', 'Superficie en km2', '_merge', 'flag_poblacion_faltante', 'tasa_delitos_propiedad_100k_v2', 'densidad_poblacion', 'tasa_lag1', 'Indice_IPC_lag1', 'variacion_anual_cbt_lag1', 'variacion_anual_salarios_lag1', 'tasa_yoy', 'log_densidad']


(4072, 34)

In [101]:
# Verificar nulos en df_mapa_sup_final
null_counts = df_mapa_sup_final.isna().sum().sort_values(ascending=False)
null_ratios = (df_mapa_sup_final.isna().mean() * 100).sort_values(ascending=False)

print(null_counts)
print("\nPorcentaje de nulos por columna:")
print(null_ratios)

tasa_yoy                          536
tasa_lag1                         530
variacion_anual_salarios_lag1     523
Indice_IPC_lag1                   523
variacion_anual_cbt_lag1          523
tasa_delitos_propiedad_100k_v2      8
log_densidad                        8
poblacion                           8
Superficie en km2                   8
densidad_poblacion                  8
provincia_nombre                    0
provincia_id                        0
departamento_id                     0
departamento_nombre                 0
poblacion_2022                      0
tasa_delitos_propiedad_100k         0
anio                                0
delitos_propiedad_hechos            0
Indice_IPC                          0
variacion_acceso_internet_pct       0
jurisdiccion                        0
variacion_anual_ipim                0
mes                                 0
variacion_empleo_const_pct          0
variacion_anual_cbt                 0
variacion_anual_salarios_pct        0
region      